# Strategic Sourcing Intelligence — FY26 empirical analysis
Real World Bank procurement award data. **Awards are not actual spend, unit prices or delivery performance.**

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from src.pipeline import sensitivity
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
df = pd.read_csv(ROOT / "data/raw/world_bank_awards.csv")
df.shape

## 1. Audit schema and missingness
Missing values are not zero. Repeated selection numbers are not automatically duplicate contracts.

In [ ]:
display(df.isna().sum().to_frame("missing"))
print("Exact duplicate rows:", df.duplicated().sum())
print("Repeated non-null selection numbers:", df["Selection Number"].dropna().duplicated().sum())

## 2. Focus on the complete FY26 disclosure regime
The published reporting threshold changed in Q3 FY25. FY26 is July 2025 through June 2026.

In [ ]:
df["date"] = pd.to_datetime(df["Award Date"], format="%m/%d/%Y")
df["value"] = pd.to_numeric(df["Contract Award Amount (USD)"], errors="coerce")
df["supplier_clean"] = df["Supplier"].str.strip().str.casefold().str.replace(r"\s+", " ", regex=True)
fy = df[df.date.between("2025-07-01", "2026-06-30")].copy()
valid = fy[fy.value.notna() & fy.value.gt(0)].copy()
print("FY26 rows:",len(fy),"usable:",len(valid),"missing amounts:",fy.value.isna().sum())

## 3. Category exposure
Rank categories by published award value. These are **not** comparable unit prices.

In [ ]:
category = valid.groupby("Commodity Category").agg(awards=("value","size"),awarded_usd=("value","sum"),suppliers=("supplier_clean","nunique")).sort_values("awarded_usd",ascending=False)
category["share"] = category.awarded_usd / category.awarded_usd.sum()
display(category.head(12))
category.head(10).awarded_usd.sort_values().div(1e6).plot.barh(figsize=(9,5),title="FY26 disclosed award value by category (USD m)"); plt.show()

## 4. Supplier concentration
HHI = 10,000 × sum of squared supplier award-value shares. High HHI is a screening flag, not evidence of supplier failure.

In [ ]:
rows=[]
for cat,g in valid.groupby("Commodity Category"):
    values=g.groupby("supplier_clean").value.sum()
    shares=values/values.sum()
    rows.append({"category":cat,"awards":len(g),"suppliers":len(values),"awarded_usd":values.sum(),"top1_share":shares.max(),"hhi":10000*(shares**2).sum()})
metrics=pd.DataFrame(rows).sort_values("awarded_usd",ascending=False)
display(metrics.head(12))

## 5. Decision scenario, not an optimization claim
Reallocate part of the top supplier’s award value to hypothetical new suppliers. This does not establish feasibility or cost savings.

In [ ]:
chosen=metrics.query("awards >= 10 and suppliers >= 3").sort_values("hhi",ascending=False).iloc[0].category
s=valid.loc[valid["Commodity Category"].eq(chosen)].groupby("supplier_clean").value.sum()
scenarios=sensitivity(s/s.sum(),fractions=(.1,.2,.3),new_counts=(1,2,3))
print("Category:",chosen)
display(scenarios)

## 6. Manager challenge
What evidence would you require before proposing a real supplier switch? Consider scope comparability, supplier capacity, service levels, switching costs, contract restrictions, and qualification risk.

**Limitations:** No delivery timestamps, quantity/unit prices, realized savings, supplier capacities, or reliable corporate-parent IDs. A lower HHI is not proof of lower operational risk.